# Checking Multicollinearity Among Predictors

This notebook measures what is the extent of multicollinearity among the predictor variables we used in our analysis. Find [here](https://online.stat.psu.edu/stat462/node/180/) more information about multicollinarity and its effect on regression analyses.

## Imports and Loading

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

In [3]:
tsv_path = "../survey_variables.tsv"
df_clean = pd.read_csv(tsv_path, sep='\t', encoding='utf-8')
print(df_clean.columns)
display(df_clean.head())

Index(['StartDate', 'EndDate', 'Status', 'Progress', 'Duration (in seconds)',
       'Finished', 'RecordedDate', 'ResponseId', 'DistributionChannel',
       'UserLanguage',
       ...
       'LT_exp', 'lt_lit01', 'lt_lit', 'chatbot_user', 'InfoRetrieval_freq',
       'ProblemSolving_freq', 'Learning_freq', 'ContentCreation_freq',
       'Entertainment_freq', 'Creativity_freq'],
      dtype='object', length=137)


,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,DistributionChannel,UserLanguage,...,LT_exp,lt_lit01,lt_lit,chatbot_user,InfoRetrieval_freq,ProblemSolving_freq,Learning_freq,ContentCreation_freq,Entertainment_freq,Creativity_freq
0,2025-05-23 2:27:17,2025-05-23 2:32:43,IP Address,100,325,True,2025-05-23 2:32:43,R_2ZE9jb0FUPux0tt,anonymous,IT,...,1,2.571429,0.642857,Yes,2.0,1.0,2.0,2.0,0.0,0.0
1,2025-05-23 2:33:42,2025-05-23 2:39:23,IP Address,100,341,True,2025-05-23 2:39:24,R_8CT67Tz9qI28TCL,anonymous,IT,...,2,3.714286,0.928571,Yes,3.0,3.0,3.0,3.0,3.0,3.0
2,2025-05-23 2:49:11,2025-05-23 2:57:29,IP Address,100,497,True,2025-05-23 2:57:29,R_2CECSpMrA1A2hEd,anonymous,IT,...,2,2.285714,0.571429,Yes,2.0,2.0,3.0,1.0,0.0,0.0
3,2025-05-23 2:51:57,2025-05-23 2:58:04,IP Address,100,366,True,2025-05-23 2:58:04,R_8f93aWfiSbk9k6u,anonymous,IT,...,2,0.000000,0.000000,Yes,2.0,2.0,3.0,2.0,1.0,3.0
4,2025-05-23 2:57:43,2025-05-23 3:02:41,IP Address,100,297,True,2025-05-23 3:02:41,R_8E5HNWdWJ4Pra9z,anonymous,IT,...,2,2.142857,0.535714,Yes,1.0,2.0,0.0,2.0,1.0,1.0


In [4]:
predictor_variables = [
    'EducationGroup', 'GeographyGroup', 'GenderGroup', 'IncomeGroup', 'AgeGroup', 'LT_exp', 'lt_lit'
]
predictors_df = df_clean[predictor_variables]
display(predictors_df)
print(predictors_df.dtypes)
print(predictors_df.shape)

,EducationGroup,GeographyGroup,GenderGroup,IncomeGroup,AgeGroup,LT_exp,lt_lit
0,Graduates,North,Woman,Higher,18-34,1,0.642857
1,Graduates,North,Man,Lower,18-34,2,0.928571
2,Graduates,North,Man,Lower,18-34,2,0.571429
3,Non-graduates,North,Woman,Lower,35-54,2,0.000000
4,Graduates,North,Man,NaN,18-34,2,0.535714
...,...,...,...,...,...,...,...
1910,Non-graduates,North,Man,Mid,65+,3,0.500000
1911,Graduates,North,Woman,Lower,65+,1,0.321429
1912,Graduates,North,Woman,Mid,65+,4,0.321429
1913,Graduates,North,Woman,Lower,65+,1,0.250000


EducationGroup     object
GeographyGroup     object
GenderGroup        object
IncomeGroup        object
AgeGroup           object
LT_exp              int64
lt_lit            float64
dtype: object
(1915, 7)


In [5]:
# drop rows with missing values
print("Shape before dropping NA:", predictors_df.shape)
predictors_df = predictors_df.dropna()
print("Shape after dropping NA:", predictors_df.shape)

# remap ordinal variables to integer encoding
predictors_df["IncomeGroup"] = predictors_df["IncomeGroup"].map({"Lower": 1, "Mid": 2, "Higher": 3})
predictors_df["AgeGroup"] = predictors_df["AgeGroup"].map({"18-34": 1, "35-54": 2, "55-64": 3, "65+": 4})

# one-hot encoding of categorical variables
predictors_df = pd.get_dummies(predictors_df, columns=["EducationGroup", "GeographyGroup", "GenderGroup"], drop_first=True)
display(predictors_df)

# Print dtypes of the final columns
print(predictors_df.dtypes)

Shape before dropping NA: (1915, 7)
Shape after dropping NA: (1861, 7)


/tmp/ipykernel_3324517/2846505158.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  predictors_df["IncomeGroup"] = predictors_df["IncomeGroup"].map({"Lower": 1, "Mid": 2, "Higher": 3})
/tmp/ipykernel_3324517/2846505158.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  predictors_df["AgeGroup"] = predictors_df["AgeGroup"].map({"18-34": 1, "35-54": 2, "55-64": 3, "65+": 4})


,IncomeGroup,AgeGroup,LT_exp,lt_lit,EducationGroup_Non-graduates,GeographyGroup_Centre,GeographyGroup_North,GeographyGroup_South and Islands,GenderGroup_Neither,GenderGroup_Woman
0,3,1,1,0.642857,False,False,True,False,False,True
1,1,1,2,0.928571,False,False,True,False,False,False
2,1,1,2,0.571429,False,False,True,False,False,False
3,1,2,2,0.000000,True,False,True,False,False,True
5,1,1,3,0.392857,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...
1910,2,4,3,0.500000,True,False,True,False,False,False
1911,1,4,1,0.321429,False,False,True,False,False,True
1912,2,4,4,0.321429,False,False,True,False,False,True
1913,1,4,1,0.250000,False,False,True,False,False,True


IncomeGroup                           int64
AgeGroup                              int64
LT_exp                                int64
lt_lit                              float64
EducationGroup_Non-graduates           bool
GeographyGroup_Centre                  bool
GeographyGroup_North                   bool
GeographyGroup_South and Islands       bool
GenderGroup_Neither                    bool
GenderGroup_Woman                      bool
dtype: object


In [6]:
def calculate_gvif(df):
    """
    Calculates the Generalized Variance Inflation Factor (GVIF) for a DataFrame.

    The function handles numerical, categorical (binary/multiclass), and ordinal variables.
    Categorical variables are expected to be one-hot encoded, with original
    variable names joined by an underscore (e.g., 'color_blue', 'color_red').
    Ordinal variables should be integer-encoded.

    Args:
        df (pd.DataFrame): The input DataFrame with predictor variables.

    Returns:
        pd.DataFrame: A DataFrame with GVIF and degrees of freedom for each variable.
    """
    gvif_data = {}
    
    # Identify unique variables from one-hot encoded columns
    # e.g., 'color_blue' and 'color_red' belong to 'color'
    def get_base_variable(col, df_cols):
        if '_' in col:
            base_name = col.split('_')[0]
            related_cols = [c for c in df_cols if c.startswith(base_name + '_')]
            if len(related_cols) > 1:
                return base_name
        return col

    variables = sorted(list(set(get_base_variable(c, df.columns) for c in df.columns)))

    print("Variables considered for GVIF calculation:", variables)

    for var in variables:
        # Get all columns related to the current variable (for one-hot encoding)
        is_categorical = False
        target_cols = [c for c in df.columns if c.startswith(var + '_')]
        if not target_cols:
            target_cols = [var]
        else:
            is_categorical = True

        # print("Target cols:", target_cols)
        # print("Is categorical:", is_categorical)
            
        y = df[target_cols]
        X = df.drop(columns=target_cols)
        X = sm.add_constant(X, has_constant='add') # Add intercept
        
        # Determine the type of the target variable
        dtype = df[target_cols[0]].dtype
        print("Dtype of current variable:", dtype)
        
        try:
            # Ensure design matrix is purely numeric (statsmodels breaks with mixed dtypes -> object ndarray)
            X_numeric = X.copy()
            for c in X_numeric.columns:
                if X_numeric[c].dtype == bool:
                    X_numeric[c] = X_numeric[c].astype(int)
            # Cast everything to float for homogeneity
            X_numeric = X_numeric.astype(float)

            # Use a 1D series when single-column target (many statsmodels classes expect 1D endog)
            # y_series = y.iloc[:, 0] if y.shape[1] == 1 else y
            y_series = y

            # Numerical target (single column, continuous)
            if np.issubdtype(dtype, np.number) and not is_categorical and df[target_cols[0]].nunique() > 2:
                model = sm.OLS(y_series.astype(float), X_numeric)
                rsquared = model.fit().rsquared
                df_deg = 1  # Degrees of freedom
            # Ordinal target (assumed if integer and not one-hot)
            elif np.issubdtype(dtype, np.integer) and not is_categorical:
                model = OrderedModel(y_series.astype(int), X_numeric, distr='logit')
                rsquared = model.fit(method='bfgs', disp=False).prsquared
                df_deg = 1
            # Categorical target (one-hot encoded or binary dummy)
            else:
                if y.shape[1] == 1:  # single binary / dummy column
                    endog = y_series.astype(int)
                    model = sm.Logit(endog, X_numeric)
                    rsquared = model.fit(disp=False).prsquared
                    df_deg = 1
                else:
                    # Multiclass one-hot (k columns) -> treat as multinomial
                    # Convert to a single categorical series (argmax of dummies)
                    endog = y.values.argmax(axis=1)
                    model = sm.MNLogit(endog, X_numeric)
                    rsquared = model.fit(disp=False).prsquared
                    df_deg = y.shape[1] - 1  # k-1
            
            gvif = 1 / (1 - rsquared)
            # GVIF^(1/(2*df)) is a scaled version for better comparison
            gvif_scaled = gvif**(1 / (2 * df_deg))
            
            gvif_data[var] = {
                "GVIF": gvif,
                "Df": df_deg,
                "GVIF^(1/(2*Df))": gvif_scaled
            }
        except Exception as e:
            print(f"Could not calculate GVIF for {var}: {e}")
            
    return pd.DataFrame.from_dict(gvif_data, orient='index')

# --- Example Usage ---

# # 1. Create a sample DataFrame
# data = {
#     'age': [25, 30, 35, 40, 45, 50, 55, 60],
#     'income': [50000, 60000, 75000, 90000, 110000, 130000, 150000, 180000],
#     'experience': [2, 5, 8, 12, 15, 18, 22, 25], # Highly correlated with age
#     'education': [2, 3, 3, 4, 4, 5, 5, 5], # Ordinal: 1=HS, 2=Bach, 3=Mast, 4=PhD
#     'city': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'B'] # Categorical
# }
# df_sample = pd.DataFrame(data)

# # 2. One-hot encode the categorical variable
# df_processed = pd.get_dummies(df_sample, columns=['city'], drop_first=True)

# # 3. Calculate and print GVIF
# gvif_results = calculate_gvif(df_processed)
# print(gvif_results)

gvif_results = calculate_gvif(predictors_df)
print(gvif_results)

Variables considered for GVIF calculation: ['AgeGroup', 'EducationGroup_Non-graduates', 'GenderGroup', 'GeographyGroup', 'IncomeGroup', 'LT_exp', 'lt_lit']
Dtype of current variable: int64
Dtype of current variable: bool
Dtype of current variable: bool
Dtype of current variable: bool
Dtype of current variable: int64
Dtype of current variable: int64
Dtype of current variable: float64
                                  GVIF  Df  GVIF^(1/(2*Df))
AgeGroup                      1.178397   1         1.085540
EducationGroup_Non-graduates  1.076732   1         1.037657
GenderGroup                   1.032715   1         1.016226
GeographyGroup                1.013202   2         1.003284
IncomeGroup                   1.073610   1         1.036152
LT_exp                        1.129561   1         1.062808
lt_lit                        1.167353   1         1.080441


In [8]:
latex_table = gvif_results.to_latex(index=True, float_format="%.3f")
print(latex_table)

\begin{tabular}{lrrr}
\toprule
 & GVIF & Df & GVIF^(1/(2*Df)) \\
\midrule
AgeGroup & 1.178 & 1 & 1.086 \\
EducationGroup_Non-graduates & 1.077 & 1 & 1.038 \\
GenderGroup & 1.033 & 1 & 1.016 \\
GeographyGroup & 1.013 & 2 & 1.003 \\
IncomeGroup & 1.074 & 1 & 1.036 \\
LT_exp & 1.130 & 1 & 1.063 \\
lt_lit & 1.167 & 1 & 1.080 \\
\bottomrule
\end{tabular}

